# Car Garage: Survey and Car Times

AI A = Close AI, AI B = Distant AI. Run this notebook from our project folder with the `data` folder.

In [ ]:
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from pathlib import Path
assert Path("data/survey.csv").exists(), "Run from the original repository root"

## 1. Load the survey

Skip the two extra rows from Qualtrics. Keep finished surveys that took more than 100 seconds, like in the earlier analysis.

In [ ]:
raw = pd.read_csv("data/survey.csv")
survey = raw.iloc[2:].copy()
survey = survey[(pd.to_numeric(survey["Duration (in seconds)"], errors="coerce") > 100) & (survey["Finished"].astype(str) == "1")].reset_index(drop=True)
print("Valid participants:", len(survey))
print("A/B preference codes:", survey["Q7.2"].value_counts().to_dict())

## 2. Compare the ratings

Q4 is for A (Close AI), and Q6 is for B (Distant AI). We compare each participant's answers. We don't swap them based on Q3.2.

In [ ]:
labels = ["Enjoyment", "Competence", "Effectiveness", "Coordination", "Adaptation"]
rows = []
for i, label in enumerate(labels, start=2):
    a = pd.to_numeric(survey[f"Q4.{i}_1"], errors="coerce")
    b = pd.to_numeric(survey[f"Q6.{i}_1"], errors="coerce")
    paired = pd.DataFrame({"close": a, "distant": b}).dropna()
    w = stats.wilcoxon(paired["distant"] - paired["close"])
    rows.append({"measure": label, "n": len(paired), "close_mean": paired.close.mean(), "distant_mean": paired.distant.mean(), "difference": (paired.distant - paired.close).mean(), "p_unadjusted": w.pvalue, "p_bonferroni": min(1, w.pvalue * len(labels))})
ratings = pd.DataFrame(rows)
print(ratings.to_string(index=False))

## 3. Check car times

Leave out the tutorial and find each person's average time for each AI.

In [ ]:
times = pd.read_csv("data/car-completion-durations.csv")
experiment = times[times["AI Type"].isin(["close", "distant"])]
means = experiment.groupby(["Participant Idx", "AI Type"])["Time"].mean().unstack().dropna()
print("Experimental cars:", len(experiment))
print("Condition means:", means.mean().to_dict())
print("Paired t-test:", stats.ttest_rel(means.close, means.distant))
print("Paired Wilcoxon:", stats.wilcoxon(means.close, means.distant))

## 4. Make graphs

In [ ]:
ax = ratings.plot(x="measure", y=["close_mean", "distant_mean"], kind="bar", figsize=(9,4), rot=15)
ax.set_ylabel("Mean rating (1–7)")
ax.set_ylim(0,7)
plt.tight_layout()
plt.show()
ax = means.plot(kind="line", marker="o", figsize=(7,4))
ax.set_ylabel("Mean car completion time (seconds)")
plt.tight_layout()
plt.show()